# Grating coupler, ring, photodetector

Place each part in front of the port that feeds it
(`chain_ports`). That keeps the figure compact and the
waveguide a short hop. Route with `path_type="smooth"`.
List optical ports that nobody connected.


In [1]:
from optodraw import Scene

def half_size(component):
    probe = Scene(title="", width=2000, height=2000, platform="on_chip")
    probe.add(component)
    x0, y0, x1, y1 = probe.content_bounds()
    return 0.5 * (x1 - x0), 0.5 * (y1 - y0)


In [2]:
def dangling_optical(scene, connected):
    used = {(id(comp), name) for comp, name in connected}
    leftover = []
    for component in scene.walk():
        ports = getattr(component, "ports", None)
        if not callable(ports):
            continue
        for name, port in ports().items():
            if getattr(port, "kind", "optical") in ("electrical", "fiber"):   # a fibre port is the off-chip side
                continue
            if name.startswith(("e", "h", "gnd", "sig")):
                continue
            if (id(component), name) in used:
                continue
            label = getattr(component, "label", None) or component.__class__.__name__
            leftover.append(f"{label}:{name}")
    return leftover


In [3]:
from optodraw import Scene, VerticalGratingCoupler, MicroringResonator, OnChipPhotodetector, Terminator
from optodraw.layout import chain_ports

print("PD half size", half_size(OnChipPhotodetector(label="PD")))
HOP = 20.0


PD half size (np.float64(33.0), np.float64(15.0))


In [4]:
scene = Scene(title="GC, ring, photodetector", width=320, height=140, platform="on_chip")
gc = scene.add(VerticalGratingCoupler(center=(0, 0), angle=180, label="GC"))
ring = scene.add(MicroringResonator(add_drop=True, label="Ring"))
pd = scene.add(OnChipPhotodetector(angle=180, label="PD"))        # on the drop, west: faces the ring
term = scene.add(Terminator(label="term"))
add = scene.add(Terminator(label="add"))
chain_ports(
    [(gc, "o1", ring, "o1"), (ring, "o3", pd, "o1"), (ring, "o2", term, "o1"), (ring, "o4", add, "o1")],
    gap=HOP, grid=0,
)
scene.route((gc, "o1"), (ring, "o1"), kind="waveguide", path_type="smooth")
scene.route((ring, "o3"), (pd, "o1"), kind="waveguide", path_type="smooth")
scene.route((ring, "o2"), (term, "o1"), kind="waveguide", path_type="smooth")
scene.route((ring, "o4"), (add, "o1"), kind="waveguide", path_type="smooth")
leftover = dangling_optical(
    scene,
    [(gc, "o1"), (ring, "o1"), (ring, "o2"), (ring, "o3"), (ring, "o4"), (pd, "o1"), (term, "o1"), (add, "o1")],
)
print("unconnected", leftover)
scene.fit_frame(margin=16)


unconnected []


In [5]:
print("bounds", scene.content_bounds())
scene.preview(show_3d=False)


bounds (np.float64(-45.0), np.float64(-6.3000000000000025), np.float64(103.8), np.float64(42.20000000000001))


ScenePreview('GC, ring, photodetector', 900x520, 3d=off)